# Representative Electric Full Correlation EDA

대표 전기 계량기 11개에 대해 raw DB 기준 전체 컬럼 correlation matrix를 확인하는 노트북입니다.

- 6년 전체
- 연도별
- 계절별

기본 스크립트 `scripts/eda_raw_correlation_representative_electric.py`의 helper를 재사용합니다.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'raw_eda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.eda_raw_correlation_representative_electric import (
    REPRESENTATIVE_METERS,
    SEASON_ORDER,
    add_period_columns,
    build_corr_matrix,
    build_engine,
    fetch_meter_raw_df,
    fetch_weather_df,
    select_usable_columns,
)

pio.renderers.default = 'notebook_connected'

In [ ]:
REPRESENTATIVE_METERS

In [ ]:
engine = build_engine()
weather_df, weather_columns = fetch_weather_df(engine)
weather_columns

In [ ]:
def load_meter_df(meter_urn: str):
    raw_df, measurements = fetch_meter_raw_df(engine, meter_urn)
    merged_df = raw_df.merge(weather_df, on='ts', how='left')
    merged_df = add_period_columns(merged_df)
    candidate_columns = measurements + weather_columns
    usable_columns = select_usable_columns(merged_df, candidate_columns)
    return merged_df, measurements, usable_columns


def plot_full_corr(df: pd.DataFrame, columns: list[str], title: str, html_name: str | None = None):
    corr_df = build_corr_matrix(df, columns)
    fig = go.Figure(
        data=go.Heatmap(
            z=corr_df.values,
            x=columns,
            y=columns,
            zmin=-1,
            zmax=1,
            colorscale='RdBu',
            reversescale=True,
            colorbar=dict(title='corr'),
            text=corr_df.round(2).values,
            texttemplate='%{text}',
            textfont=dict(size=9),
            hovertemplate='x=%{x}<br>y=%{y}<br>corr=%{z:.4f}<extra></extra>',
        )
    )
    fig.update_layout(
        title=title,
        width=max(900, len(columns) * 36),
        height=max(800, len(columns) * 32),
        xaxis=dict(tickangle=90),
        yaxis=dict(autorange='reversed'),
    )
    if html_name is not None:
        fig.write_html(OUTPUT_DIR / html_name, include_plotlyjs='cdn')
    fig.show()
    return corr_df


def show_meter_6year(meter_urn: str):
    merged_df, measurements, usable_columns = load_meter_df(meter_urn)
    print('meter_urn =', meter_urn)
    print('measurement columns =', measurements)
    print('usable column count =', len(usable_columns))
    print('usable columns =', usable_columns)
    return plot_full_corr(merged_df, usable_columns, f'{meter_urn} 6-Year Full Correlation Matrix', f'{meter_urn}_6year_full_corr.html')


def show_meter_year(meter_urn: str, year: int):
    merged_df, _, usable_columns = load_meter_df(meter_urn)
    year_df = merged_df.loc[merged_df['year'] == year].copy()
    year_columns = select_usable_columns(year_df, usable_columns)
    print('meter_urn =', meter_urn, '| year =', year)
    print('rows =', len(year_df))
    print('usable columns =', year_columns)
    return plot_full_corr(year_df, year_columns, f'{meter_urn} {year} Full Correlation Matrix', f'{meter_urn}_{year}_full_corr.html')


def show_meter_season(meter_urn: str, season: str):
    merged_df, _, usable_columns = load_meter_df(meter_urn)
    season_df = merged_df.loc[merged_df['season'] == season].copy()
    season_columns = select_usable_columns(season_df, usable_columns)
    print('meter_urn =', meter_urn, '| season =', season)
    print('rows =', len(season_df))
    print('usable columns =', season_columns)
    return plot_full_corr(season_df, season_columns, f'{meter_urn} {season} Full Correlation Matrix', f'{meter_urn}_{season}_full_corr.html')

## Example: 6-Year Full Matrix

In [ ]:
corr_6year = show_meter_6year('H1.Z16')
corr_6year.round(3)

## Example: Yearly Full Matrix

In [ ]:
corr_2023 = show_meter_year('H1.Z16', 2023)
corr_2023.round(3)

## Example: Seasonal Full Matrix

In [ ]:
corr_summer = show_meter_season('H1.Z16', 'Summer')
corr_summer.round(3)

## Batch Run Template

In [ ]:
TARGET_METER = 'H1.Z16'
TARGET_YEAR = 2023
TARGET_SEASON = 'Summer'

show_meter_6year(TARGET_METER)
show_meter_year(TARGET_METER, TARGET_YEAR)
show_meter_season(TARGET_METER, TARGET_SEASON)